# Explainability and Clinical Alerts

In this notebook, we interpret the best performing model (e.g., LightGBM for the 24h prediction window) using SHAP (SHapley Additive exPlanations). 

Our explainability strategy aims to provide:
1. **Global insights:** Understanding which features drive the model's predictions overall (e.g., feature importance, SHAP summary plots).
2. **Local explanations:** Providing patient-specific insights to understand why a particular patient is flagged at high risk.
3. **Clinical Alerts:** Generating structured JSON alerts that highlight the top risk factors, which can be directly integrated into an EHR (Electronic Health Record) system to guide clinical decision-making.

In [ ]:
import os
import sys
import json
import shap
import pandas as pd
import matplotlib.pyplot as plt

# Add parent directory to path to allow importing from src
sys.path.append('..')
from src.explain import run_explainability

# Configure paths for model, data, and outputs
model_path = '../models/24h_lgbm_model.pkl'
data_path = '../data/processed/features_24h.csv'
output_dir = '../reports/figures/explainability'
alerts_path = '../reports/alerts/clinical_alerts.json'

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.dirname(alerts_path), exist_ok=True)

print("Libraries loaded and environment configured.")

In [ ]:
print("Running explainability pipeline...")

# The run_explainability function will:
# - Load the trained LightGBM model and processed dataset
# - Calculate SHAP values for the test cohort
# - Generate and save SHAP summary plots and patient-specific waterfall plots
# - Extract top risk factors for high-risk patients and save them as JSON alerts

run_explainability(
    model_path=model_path,
    data_path=data_path,
    output_dir=output_dir,
    alerts_path=alerts_path
)

print(f"Explainability analysis complete. Visualizations saved to: {output_dir}")
print(f"Clinical alerts saved to: {alerts_path}")

## Sample Clinical Alert

To show what the EHR would display, we can print a sample alert from the JSON file generated above. The alert includes a risk score and a breakdown of the specific factors driving that risk, providing actionable context to the clinician.

In [ ]:
# Load and print a sample alert to simulate EHR display
try:
    with open(alerts_path, 'r') as f:
        alerts = json.load(f)
        
    if alerts:
        sample_alert = alerts[0]
        
        print("=" * 50)
        print("🚨 EHR CLINICAL ALERT 🚨")
        print("=" * 50)
        print(f"Patient ID : {sample_alert.get('patient_id', 'Unknown')}")
        print(f"Risk Score : {sample_alert.get('risk_score', 0.0):.2f}")
        print("\nTop Risk Factors Contributing to Alert:")
        
        for factor, impact in sample_alert.get('top_features', {}).items():
            print(f" ⚠️ {factor:20s} (+{impact:.3f})")
        
        print("=" * 50)
    else:
        print("No alerts were generated.")
        
except FileNotFoundError:
    print("Alerts file not found. Ensure `run_explainability` successfully generated the JSON file.")